# CO₂ report, 2024 — as it was handed over

A colleague wrote this notebook for a briefing note. The briefing needs three numbers:

1. how much CO₂ the world emitted in 2024,
2. China's share of it,
3. how many countries reported emissions that year.

The data is Our World in Data's CO₂ file, `data/raw/owid_co2.csv` (what each column means: `data/raw/owid_codebook.csv`).
The cells under **The report** are your colleague's. They run without an error. Run them, read the three numbers, and
then read the README: something in this same file disagrees with them.

*(The other file in `data/raw/`, `online_retail.parquet`, is the one the lecture used. This lab does not need it.)*

In [ ]:
import os
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_rows", 400)   # up to 400 rows in full; a longer result prints head and tail with "..." between: count it, then census it by group

# Anchor to the project folder (DS1, Block 1), then stand there: every path below is from the project folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
print("Working in:", Path.cwd())   # must print this project's folder

con = duckdb.connect()             # an in-memory database; it reads the files in data/raw/ directly

## The report

### 1. World CO₂ emissions in 2024 (million tonnes)

In [ ]:
con.sql("""
    SELECT SUM(co2) AS world_co2_mt
    FROM 'data/raw/owid_co2.csv'
    WHERE year = 2024
""").df()

### 2. China's share of world emissions in 2024

In [ ]:
world = con.sql("SELECT SUM(co2) FROM 'data/raw/owid_co2.csv' WHERE year = 2024").fetchone()[0]
china = con.sql("SELECT co2 FROM 'data/raw/owid_co2.csv' WHERE country = 'China' AND year = 2024").fetchone()[0]
print(f"China emitted {china:,.1f} Mt of {world:,.1f} Mt: a share of {china / world:.1%}")

### 3. How many countries reported emissions in 2024

In [ ]:
con.sql("""
    SELECT COUNT(*) AS countries
    FROM 'data/raw/owid_co2.csv'
    WHERE year = 2024
""").df()

---

# Your work starts here

Work top to bottom. Paste what each inspection returns into `DIAGNOSIS.md`, part 3, **before** you change anything.

## A. Inspect the table (the lecture's four recipes — copy them, then read what they return)

Run each one on `data/raw/owid_co2.csv`. For each, write one sentence in a comment: what did it tell you?

In [ ]:
# 1. What columns, and what types did DuckDB infer?
con.sql("DESCRIBE SELECT * FROM 'data/raw/owid_co2.csv'").df()

In [ ]:
# 2. One row per column: its type, how many values are missing, its smallest and largest value.
con.sql("SUMMARIZE SELECT * FROM 'data/raw/owid_co2.csv'").df()

In [ ]:
# 3. The key test. The codebook says one row is one country in one year. Is (country, year) unique?
con.sql("""
    SELECT COUNT(*) AS n_rows, COUNT(DISTINCT (country, year)) AS n_distinct_keys
    FROM 'data/raw/owid_co2.csv'
""").df()

In [ ]:
# 4. The census of the country column for one year: every value, never just the top few.
con.sql("""
    SELECT country, iso_code, co2
    FROM 'data/raw/owid_co2.csv'
    WHERE year = 2024
    ORDER BY co2 DESC NULLS LAST
""").df()

## B. Which rows are countries?

Write the filter as a **view**: a saved query with a name. Every corrected query below reads `FROM countries`, so the
filter is written once and can be checked once. Replace the comment with your `CREATE OR REPLACE VIEW` statement.

In [ ]:
# con.sql("""
#     CREATE OR REPLACE VIEW countries AS
#     SELECT *
#     FROM 'data/raw/owid_co2.csv'
#     WHERE ...        -- your filter: which rows are countries?
# """)

Now run the census again **on your view**, for 2024. A filter is a claim: read every row it kept.

In [ ]:
# The census, again, but FROM countries.

## C. The ten largest emitting countries in 2024

Write this query yourself, from an empty cell: country and CO₂, the ten largest, largest first.

In [ ]:
# Your query.

## D. The check: does the world add up?

The file has its own answer for the whole world: the row whose `country` is `World`. Put the sum over **your** view
beside it. Our World in Data's documentation says `World` also contains emissions that belong to no country —
international aviation and international shipping — which are rows of their own in this file. So:

> World = the countries + International aviation + International shipping, to the rounding.

Compute the residual, World minus the sum over your view. Then subtract aviation and shipping. What is left must be
smaller than **0.01 Mt**: every value in this file is rounded to 0.001 Mt, and a few hundred such roundings add up to
a few thousandths. Anything larger is a row: your filter kept something that is not a country, or lost one.

The sum is not proof of membership. A row worth 0.000 Mt in 2024 passes through it unseen, and so would the three
smallest countries. The proof that your view holds exactly the countries is the census you read in section B.

In [ ]:
# World row, the sum over your view, the residual, aviation, shipping — and what is left.

## E. The corrected report

The three numbers again, from your view: the world total from the `World` row, China's share of it, and the number
of countries that reported (a country "reported" if it has a value for `co2`).

In [ ]:
# The three corrected numbers.

## F. Hand-off to Block 2 (run this last; it is supplied)

Block 2 starts from the table you just built: every country, every year. This cell writes it to `data/silver/`.
`data/silver/` is ignored by Git: it is made by this notebook, so it is never committed.

In [ ]:
# Supplied. Uncomment and run once your view exists.
# os.makedirs("data/silver", exist_ok=True)
# con.sql("COPY (SELECT * FROM countries ORDER BY country, year) TO 'data/silver/countries.csv' (HEADER)")
# print("wrote data/silver/countries.csv:", con.sql("SELECT COUNT(*) FROM countries").fetchone()[0], "rows")